<a href="https://colab.research.google.com/github/anomalyco/opencode/blob/main/denoising_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Diffusion-based Tabular Data Denoising

This notebook demonstrates how diffusion models can be applied to denoise tabular data, using credit card transaction data as an example.

## Overview

Diffusion models work by learning to reverse a noise process. In the context of tabular data, we can apply this concept by:
1. Adding controlled noise to clean data
2. Training a model to learn how to remove that noise
3. Applying the learned denoising to new noisy data

This simple demonstration focuses on the core concepts rather than building full diffusion models.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 1. Load and Preprocess Data

Let's load the credit card transaction dataset and prepare it for analysis.

In [ ]:
# Load the dataset
df = pd.read_csv('credit_card_transactions.csv')
print(f'Dataset shape: {df.shape}')
print('\nFirst few rows:')
df.head()

In [ ]:
# Select numerical columns for demonstration
# These columns are suitable for denoising techniques
numerical_cols = ['amt', 'lat', 'long', 'city_pop', 'unix_time', 'merch_lat', 'merch_long']

# Filter data and drop missing values
df_processed = df[numerical_cols].dropna()

print(f'Processed data shape: {df_processed.shape}')
print('Numerical columns selected for analysis:', numerical_cols)

df_processed.head()

## 2. Generate Noisy Data

To simulate real-world corrupted data, we'll add random noise to the clean dataset.

In [ ]:
def create_noisy_data(df, noise_level=0.3):
    '''Add controlled noise to the data'''
    df_noisy = df.copy()
    for col in df.columns:
        # Add Gaussian noise to each column
        if col in df.columns:
            noise = np.random.normal(0, noise_level * df[col].std(), len(df))
            df_noisy[col] = df[col] + noise
    
    return df_noisy

# Create noisy version of our data
df_noisy = create_noisy_data(df_processed, noise_level=0.5)

print('Noise added to dataset')
print('Original data example:')
print(df_processed.head())
print('\nNoisy data example:')
print(df_noisy.head())

## 3. Simple Diffusion Denoising Implementation

This represents a simplified version of how a diffusion model might work:
- Learn the noise pattern
- Apply smoothing technique to remove noise
- Preserve data integrity

In [ ]:
def simple_diffusion_denoising(original_data, noisy_data, steps=100):
    '''Simple denoising using smoothing techniques'''
    # This is a simplified version
    # In a real diffusion model, this would involve:
    # 1. Training a neural network to denoise step-by-step
    # 2. Reverse process from noise to clean data
    
    denoised_data = pd.DataFrame()
    
    for col in noisy_data.columns:
        # Apply a moving average filter to denoise
        if len(noisy_data) > 10:
            # Create exponentially weighted smoothing
            weights = np.exp(-np.arange(10)/5)
            weights = weights / np.sum(weights)
            
            # Apply weights
            smoothed = []
            for i in range(len(noisy_data)):
                start_idx = max(0, i - len(weights) + 1)
                end_idx = i + 1
                window = noisy_data[col].iloc[start_idx:end_idx]
                if len(window) > 0:
                    smoothed_val = np.average(window, weights=weights[-len(window):])
                    smoothed.append(smoothed_val)
                else:
                    smoothed.append(noisy_data[col].iloc[i])
            denoised_data[col] = smoothed
        else:
            denoised_data[col] = noisy_data[col]

    return denoised_data

# Apply denoising
df_denoised = simple_diffusion_denoising(df_processed, df_noisy)

print('Denoising process complete')
print('Denoised data example:')
print(df_denoised.head())

## 4. Visualize Results

Compare the original, noisy, and denoised datasets through various visualizations.

In [ ]:
def visualize_results(original_data, noisy_data, denoised_data):
    '''Create visualizations comparing datasets'''
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Distribution of amount
    axes[0,0].hist(original_data['amt'], alpha=0.5, label='Original', bins=50, color='blue')
    axes[0,0].hist(noisy_data['amt'], alpha=0.5, label='Noisy', bins=50, color='red')
    axes[0,0].hist(denoised_data['amt'], alpha=0.5, label='Denoised', bins=50, color='green')
    axes[0,0].set_title('Amount Distribution Comparison')
    axes[0,0].legend()
    axes[0,0].set_xlabel('Amount')
    axes[0,0].set_ylabel('Frequency')
    
    # Correlation matrix for original data
    corr_orig = original_data.corr()
    im = axes[0,1].imshow(corr_orig, cmap='coolwarm', aspect='auto')
    axes[0,1].set_title('Original Data Correlation Matrix')
    axes[0,1].set_xticks(range(len(corr_orig.columns)))
    axes[0,1].set_yticks(range(len(corr_orig.columns)))
    axes[0,1].set_xticklabels(corr_orig.columns, rotation=45)
    axes[0,1].set_yticklabels(corr_orig.columns)
    plt.colorbar(im, ax=axes[0,1])
    
    # Scatter plot of two key features
    axes[1,0].scatter(original_data['lat'], original_data['long'], alpha=0.3, label='Original', color='blue')
    axes[1,0].scatter(noisy_data['lat'], noisy_data['long'], alpha=0.3, label='Noisy', color='red')
    axes[1,0].set_title('Location Distribution')
    axes[1,0].set_xlabel('Latitude')
    axes[1,0].set_ylabel('Longitude')
    axes[1,0].legend()
    
    # Comparison of noisy vs denoised amount
    axes[1,1].scatter(original_data['amt'], noisy_data['amt'], alpha=0.3, label='Noisy vs Original', color='red')
    axes[1,1].scatter(original_data['amt'], denoised_data['amt'], alpha=0.3, label='Denoised vs Original', color='green')
    axes[1,1].plot([original_data['amt'].min(), original_data['amt'].max()], 
                   [original_data['amt'].min(), original_data['amt'].max()], 'k--')
    axes[1,1].set_title('Amount Comparison')
    axes[1,1].set_xlabel('Original Amount')
    axes[1,1].set_ylabel('Noisy/Denoised Amount')
    axes[1,1].legend()
    
    plt.tight_layout()
    plt.savefig('diffusion_denoising_results.png', dpi=300, bbox_inches='tight')
    plt.show()

# Create visualizations
visualize_results(df_processed, df_noisy, df_denoised)

## 5. Statistics and Evaluation

Quantitatively evaluate the effectiveness of our denoising approach.

In [ ]:
def print_statistics(original_data, noisy_data, denoised_data):
    '''Print statistics comparing the datasets'''
    
    print("=== Data Statistics ===")
    print("Original Data:")
    print(original_data.describe())
    
    print("\nNoisy Data:")
    print(noisy_data.describe())
    
    print("\nDenoised Data:")
    print(denoised_data.describe())
    
    # Calculate errors
    print("\n=== Denoising Errors ===")
    mse_noisy = np.mean((original_data.values - noisy_data.values)**2)
    mse_denoised = np.mean((original_data.values - denoised_data.values)**2)
    
    print(f'Mean Squared Error (Noisy vs Original): {mse_noisy:.4f}')
    print(f'Mean Squared Error (Denoised vs Original): {mse_denoised:.4f}')
    print(f'Improvement: {(mse_noisy - mse_denoised) / mse_noisy * 100:.2f}%')

# Print statistics
print_statistics(df_processed, df_noisy, df_denoised)

## 6. Conclusion and Next Steps

In this demonstration:

1. We loaded and preprocessed credit card transaction data
2. Added controlled noise to simulate corrupted data
3. Applied a simple denoising approach using smoothing techniques 
4. Visualized and evaluated the results

While this example uses a simplified approach, real-world diffusion models for tabular data would:

✅ Implement actual neural networks to learn denoising
✅ Use iterative denoising steps
✅ Train on large datasets
✅ Handle various data types (numerical, categorical, mixed)
✅ Apply advanced architectures like transformers or graph neural networks

To extend this work:
1. Implement proper diffusion model architectures
2. Use real tabular diffusion models like TabDDPM
3. Apply to different data types
4. Evaluate on larger datasets